# 실험 03: LangGraph 기본 - 휴먼 인 더 루프 (Human-in-the-loop)

## 1. 실험 제목과 목표
- **제목**: 결재 도장 찍기
- **목표**: 교안 15번 슬라이드의 세 번째 패턴인 **Human-in-the-loop (사람 승인 후 실행)** 패턴을 구현합니다. 메일 초안 작성 -> 사용자 승인 대기 -> 발송 또는 재작성의 워크플로우를 만듭니다.

## 2. 라이브러리 import

In [1]:
import os
from dotenv import load_dotenv
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

## 3. 환경 변수 로드 및 State 정의

In [2]:
load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

class MailState(TypedDict):
    request_text: str # 사용자의 최초 메일 작성 요청
    draft: str # 봇이 작성한 초안
    user_feedback: str # 승인 대기 중 사용자가 입력한 피드백 (비어있으면 승인으로 간주)

## 4. Node 정의
초안 작성 노드와 메일 발송 노드 2개를 만듭니다.

In [3]:
def node_draft(state: MailState):
    print("\n[Node: Draft] 메일 초안을 작성합니다.")
    req = state["request_text"]
    feedback = state.get("user_feedback", "")
    
    if feedback:
        sys_msg = SystemMessage(content="당신은 비서입니다. 사용자의 피드백을 반영하여 메일 초안을 수정해주세요.")
        user_msg = HumanMessage(content=f"원래 요청: {req}\n이전 초안: {state['draft']}\n수정 요청: {feedback}")
    else:
        sys_msg = SystemMessage(content="당신은 비서입니다. 사용자의 요청에 맞게 정중한 비즈니스 메일 초안을 작성해주세요.")
        user_msg = HumanMessage(content=req)
        
    res = llm.invoke([sys_msg, user_msg])
    
    return {"draft": res.content, "user_feedback": ""} # 피드백 초기화

def node_send(state: MailState):
    print("\n[Node: Send] 🚀 메일을 진짜로 발송합니다! (삐빅- 발송 완료)")
    return state # 상태 변경 없음

## 5. 그래프 조립 및 Guardrails 부착
중요 포인트: 발송(`node_send`) 직전에 멈추도록 설정합니다.

In [4]:
workflow = StateGraph(MailState)

workflow.add_node("draft", node_draft)
workflow.add_node("send", node_send)

workflow.set_entry_point("draft")
workflow.add_edge("draft", "send") # 드래프트 후 센드로 가라 (하지만 우리가 인터럽트를 걸 것입니다)
workflow.add_edge("send", END)

# *** 핵심 ***
# Checkpointer(상태 저장소)가 필수입니다. 멈춰있는 동안 상태가 날아가면 안 되기 때문입니다.
memory = MemorySaver()
# interrupt_before=["send"] 설정으로 인해, send 노드 진입 직전에 무조건 실행이 정지됩니다.
app = workflow.compile(checkpointer=memory, interrupt_before=["send"])

## 6. 실험 1: 실행 및 인터럽트(대기) 발생
메일 작성을 지시해봅니다.

In [5]:
# 대화방(Thread) 번호 부여
config = {"configurable": {"thread_id": "mail-thread-1"}}

request = "팀장님께 내일 연차 쓰겠다고 메일 좀 써줘. 이유는 감기몸살."

print("=== [1단계: 사용자 지시] ===")
for event in app.stream({"request_text": request}, config):
    pass

current_state = app.get_state(config)
print("\n=== [2단계: 초안 확인 및 대기 중] ===")
print("작성된 초안:\n", current_state.values["draft"])
print("\n--- [시스템 상태] ---")
print("다음 실행 예정 노드:", current_state.next)
print("사용자 승인을 기다리는 중입니다...")

=== [1단계: 사용자 지시] ===

[Node: Draft] 메일 초안을 작성합니다.

=== [2단계: 초안 확인 및 대기 중] ===
작성된 초안:
 제목: 내일 연차 요청

안녕하세요, 팀장님.

저는 내일 연차를 사용하고자 합니다. 최근에 감기몸살 증세로 인해 몸 상태가 좋지 않아 휴식을 취할 필요성이 있습니다. 

업무에 불편을 끼치지 않도록 최선을 다해 준비하겠습니다. 필요하신 사항이 있으시면 언제든지 말씀해 주시기 바랍니다.

감사합니다.

[당신의 이름]  
[당신의 직위]  
[당신의 연락처]  

--- [시스템 상태] ---
다음 실행 예정 노드: ('send',)
사용자 승인을 기다리는 중입니다...


## 7. 실험 2: 반려 및 재작성 요청
사용자가 초안이 마음에 안 들어서 수정을 지시합니다. (상태를 직접 업데이트합니다)

In [6]:
print("=== [3단계: 사용자 반려 (피드백 입력)] ===")
user_feedback = "너무 딱딱해. 좀 더 아파 보이는 느낌으로 앓는 소리를 추가해줘."
print("사용자 피드백:", user_feedback)

# 상태(State)에 피드백을 강제로 주입하고, 다음 노드(send)가 아닌 draft 노드로 '되돌려' 보냅니다.
app.update_state(config, {"user_feedback": user_feedback}, as_node="send") 
# Note: as_node="send"는 "방금 send 노드가 에러나서 되돌아온 것처럼 속여서" 
# 다시 이전 노드들을 실행하게 만드는 고급 트릭 중 하나이거나, 
# 혹은 명시적으로 edge를 타게 만드는 방법입니다. 
# 더 정석적인 방법은 아래와 같이 아예 draft 노드로 상태를 꽂는 것입니다.

# 올바른 방법: 현재 상태를 통째로 덮어쓰고, 그래프를 다시 돌립니다.
# 여기서는 단순화를 위해 app.invoke 로 draft부터 다시 태웁니다.
for event in app.stream({"user_feedback": user_feedback}, config):
    pass

current_state = app.get_state(config)
print("\n=== [4단계: 수정된 초안 확인 및 다시 대기] ===")
print("수정된 초안:\n", current_state.values["draft"])
print("\n다음 실행 예정 노드:", current_state.next)

=== [3단계: 사용자 반려 (피드백 입력)] ===
사용자 피드백: 너무 딱딱해. 좀 더 아파 보이는 느낌으로 앓는 소리를 추가해줘.

[Node: Draft] 메일 초안을 작성합니다.

=== [4단계: 수정된 초안 확인 및 다시 대기] ===
수정된 초안:
 제목: 내일 연차 요청

안녕하세요, 팀장님.

저는 내일 연차를 사용하고자 합니다. 요즘 정말 감기몸살에 시달리고 있어서 몸이 너무 힘들고, 세수할 힘조차 없네요. 이렇게 아픈 상태로는 업무를 제대로 할 수 없을 것 같아 휴식이 필요하다고 느끼고 있습니다.

업무에 불편을 끼치지 않도록 최대한 준비는 해놓겠지만, 혹시 필요한 사항이 있으면 언제든지 말씀해 주세요. 

감사합니다.

[당신의 이름]  
[당신의 직위]  
[당신의 연락처]

다음 실행 예정 노드: ('send',)


## 8. 실험 3: 최종 승인 및 발송
마음에 들었으니 승인(None을 입력하며 재개)합니다.

In [7]:
print("=== [5단계: 최종 승인!] ===")
# stream에 None을 주면 멈춰있던 곳(send 직전)부터 알아서 마저 실행합니다.
for event in app.stream(None, config):
    pass

print("\n[최종 상태 확인]")
current_state = app.get_state(config)
print("다음 실행 예정 노드:", current_state.next, "(비어있으면 END 도달)")
print("실무에서 이 구조는 '관리자 대시보드'와 연동되어 매우 강력하게 쓰입니다.")

=== [5단계: 최종 승인!] ===

[Node: Send] 🚀 메일을 진짜로 발송합니다! (삐빅- 발송 완료)

[최종 상태 확인]
다음 실행 예정 노드: () (비어있으면 END 도달)
실무에서 이 구조는 '관리자 대시보드'와 연동되어 매우 강력하게 쓰입니다.
